# ML_U2_Lab04 — Ensemble Models

**Versión:** 2025-1 | **Modificado:** 2026-04-24
**Dataset:** breast_cancer (sklearn) | **Duración:** 1 hora

---

## 📋 Instrucciones diferenciadas por audiencia

| Audiencia | Enfoque | TODOs |
|-----------|---------|-------|
| 🔵 **Pregrado** | Aplicaciones prácticas, intuición | TODO 1-3, 5-7 |
| 🟡 **Doctorado** | Teoría, derivaciones, análisis profundo | TODO 1-8 (incluye PhD) |

---

## 🗂️ Estructura: Parte | Tema | Tiempo | Audiencia

| Parte | Tema | Tiempo | Audiencia |
|-------|------|--------|----------|
| **Encabezado** | Contexto y meta | — | Todos |
| **Setup** | Imports, carga datos, split 80/20 | 5 min | Todos |
| **PARTE 1** | Ensemble a mano (voting manual) | 10 min | Todos |
| **PARTE 2** | Random Forest + feature importance | 20 min | Todos |
| **PARTE 3** | Boosting (AdaBoost, GBM, XGBoost) + comparación | 20 min | Todos |
| **FINAL** | Análisis e interpretación | 10 min | Diferenciado |
| **BONUS** | Voting y Stacking | Opcional | Diferenciado |

## ⚙️ SETUP (NO MODIFICAR)

Carga de librerías, datos y configuración inicial.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Configuración global
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Carga dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("✅ Setup completo")
print(f"   - X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"   - y_train: {y_train.sum()} positivos / {len(y_train)-y_train.sum()} negativos")
print(f"   - y_test: {y_test.sum()} positivos / {len(y_test)-y_test.sum()} negativos")
print(f"   - RANDOM_STATE = {RANDOM_STATE}")

# PARTE 1 — Ensemble a mano (Sin computador)

**Objetivo:** Entender el concepto de voting y por qué los ensembles pueden mejorar.

**Tiempo:** 10 minutos | **Audiencia:** Todos

---

## Ejercicio: Voting manual sobre 5 muestras

Dado un conjunto de **3 clasificadores** que predicen sobre **5 muestras**:

| Muestra | Real | Clf1 | Clf2 | Clf3 |
|---------|------|------|------|------|
| 1 | 1 | 1 | 1 | 0 |
| 2 | 0 | 0 | 1 | 0 |
| 3 | 1 | 1 | 1 | 1 |
| 4 | 0 | 1 | 0 | 0 |
| 5 | 1 | 0 | 1 | 1 |

### Preguntas:

In [ ]:
# VERIFICACIÓN: Tabla de datos
data_dict = {
    'Muestra': [1, 2, 3, 4, 5],
    'Real': [1, 0, 1, 0, 1],
    'Clf1': [1, 0, 1, 1, 0],
    'Clf2': [1, 1, 1, 0, 1],
    'Clf3': [0, 0, 1, 0, 1]
}

df_ensemble = pd.DataFrame(data_dict)
print("Tabla de predicciones:")
print(df_ensemble)
print()

# a) Accuracy individual
acc_clf1 = (df_ensemble['Clf1'] == df_ensemble['Real']).sum() / len(df_ensemble)
acc_clf2 = (df_ensemble['Clf2'] == df_ensemble['Real']).sum() / len(df_ensemble)
acc_clf3 = (df_ensemble['Clf3'] == df_ensemble['Real']).sum() / len(df_ensemble)

print(f"a) Accuracy individual:")
print(f"   Clf1: {acc_clf1:.2%}")
print(f"   Clf2: {acc_clf2:.2%}")
print(f"   Clf3: {acc_clf3:.2%}")
print()

# b) Hard Voting
def hard_vote(row):
    votes = [row['Clf1'], row['Clf2'], row['Clf3']]
    return 1 if votes.count(1) >= 2 else 0

df_ensemble['Hard_Vote'] = df_ensemble.apply(hard_vote, axis=1)
acc_voting = (df_ensemble['Hard_Vote'] == df_ensemble['Real']).sum() / len(df_ensemble)

print(f"b) Hard Voting:")
print(df_ensemble[['Real', 'Hard_Vote']])
print(f"   Accuracy del ensemble: {acc_voting:.2%}")
print()

# c) Comparación
print(f"c) Análisis:")
models = ['Clf1', 'Clf2', 'Clf3', 'Hard_Vote']
accuracies = [acc_clf1, acc_clf2, acc_clf3, acc_voting]
for model, acc in zip(models, accuracies):
    print(f"   {model:12s}: {acc:.2%}")

best_individual = max(acc_clf1, acc_clf2, acc_clf3)
print(f"\n   ¿El ensemble ({acc_voting:.2%}) supera al mejor individual ({best_individual:.2%})? {acc_voting > best_individual}")

# PARTE 2 — Random Forest

**Objetivo:** Entrenar RF, entender importancia de features y efecto del número de árboles.

**Tiempo:** 20 minutos | **Audiencia:** Todos

---

In [ ]:
# TODO 1: Entrenar DecisionTree y RandomForest
print("TODO 1: Entrenar DT y RF")
print("=" * 60)

dt_clf = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
dt_clf.fit(X_train, y_train)
y_pred_dt = dt_clf.predict(X_test)

rf_clf = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=RANDOM_STATE)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)

acc_dt = accuracy_score(y_test, y_pred_dt)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f"Decision Tree (max_depth=3):")
print(f"  Accuracy: {acc_dt:.4f}")
print(f"\nRandom Forest (n_estimators=100):")
print(f"  Accuracy: {acc_rf:.4f}")
print(f"\n✅ Mejora RF sobre DT: {(acc_rf - acc_dt):.4f} ({((acc_rf/acc_dt - 1)*100):+.2f}%)")
print()

In [ ]:
# TODO 2: Feature importances
print("TODO 2: Feature importances del RF")
print("=" * 60)

importances = rf_clf.feature_importances_
indices = np.argsort(importances)[-10:]  # Top 10
features_top10 = X.columns[indices]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(features_top10, importances[indices], color='steelblue', edgecolor='black')
ax.set_xlabel('Feature Importance', fontsize=12, fontweight='bold')
ax.set_ylabel('Feature', fontsize=12, fontweight='bold')
ax.set_title('Top 10 Features — Random Forest (n_estimators=100)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Top 10 features:")
for feat, imp in zip(features_top10, importances[indices]):
    print(f"  {feat:30s}: {imp:.4f}")
print()

In [ ]:
# TODO 3: Learning curve: accuracy vs n_estimators
print("TODO 3: Learning curve (Accuracy vs n_estimators)")
print("=" * 60)

n_estimators_range = [1, 5, 10, 25, 50, 100, 200]
test_accuracies = []

for n_est in n_estimators_range:
    rf_temp = RandomForestClassifier(n_estimators=n_est, max_depth=None, random_state=RANDOM_STATE)
    rf_temp.fit(X_train, y_train)
    y_pred_temp = rf_temp.predict(X_test)
    acc_temp = accuracy_score(y_test, y_pred_temp)
    test_accuracies.append(acc_temp)
    print(f"  n_estimators={n_est:3d}: accuracy={acc_temp:.4f}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(n_estimators_range, test_accuracies, 'o-', linewidth=2.5, markersize=8, color='darkgreen', label='Test Accuracy')
ax.axhline(y=acc_dt, color='red', linestyle='--', linewidth=2, label=f'DT Baseline ({acc_dt:.4f})')
ax.set_xlabel('Number of Estimators', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Learning Curve: RF Accuracy vs n_estimators', fontsize=13, fontweight='bold')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print()

In [ ]:
# TODO 4 [PhD]: OOB Score
print("TODO 4 [PhD]: Out-of-Bag (OOB) Score")
print("=" * 60)

rf_oob = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=RANDOM_STATE)
rf_oob.fit(X_train, y_train)

oob_score = rf_oob.oob_score_
test_accuracy = accuracy_score(y_test, rf_oob.predict(X_test))

print(f"OOB Score:     {oob_score:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Diferencia:    {abs(oob_score - test_accuracy):.4f}")
print()
print("Interpretación teórica:")
print("  - El OOB estimator usa muestras out-of-bag (no vistas) en cada árbol.")
print("  - Es una estimación insesgada de la generalization error.")
print("  - Si OOB ≈ Test Acc, no hay distributional shift entre train y test.")
print()

In [ ]:
# Tests de sanidad PARTE 2 (NO MODIFICAR)
print("\n" + "="*60)
print("TESTS DE SANIDAD — PARTE 2")
print("="*60)

assert X_train.shape[0] > 0 and X_test.shape[0] > 0, "❌ Shapes inválidos"
assert 0 <= acc_dt <= 1 and 0 <= acc_rf <= 1, "❌ Accuracy fuera de rango [0,1]"
assert acc_rf >= acc_dt * 0.8, "❌ RF tuvo desempeño sospechosamente peor que DT"
assert len(importances) == X.shape[1], "❌ Número de importances ≠ número de features"
assert all(acc >= 0.7 for acc in test_accuracies), "❌ Algunas accuracies muy bajas"
print("✅ Todos los tests pasaron.")

# PARTE 3 — Boosting y Comparación de Ensembles

**Objetivo:** Entrenar boosting, comparar modelos, entender regularización.

**Tiempo:** 20 minutos | **Audiencia:** Todos

---

## Boosting vs Bagging

**Bagging (Random Forest):** Entrena modelos en paralelo, cada uno en muestra independiente.

**Boosting (AdaBoost, GBM):** Entrena modelos **secuencialmente**, enfatizando en los errores anteriores.
- Cada modelo nuevo aprende de los fallos del anterior.
- Ajusta pesos de muestras para enfatizar los ejemplos difíciles.

**Ventaja:** Boosting reduce tanto **sesgo** como **varianza**.
**Desventaja:** Puede sobreajustar si se entrena demasiadas iteraciones.

In [ ]:
# TODO 5: Entrenar AdaBoost y GradientBoosting
print("TODO 5: Entrenar AdaBoost y GradientBoosting")
print("=" * 60)

ada_clf = AdaBoostClassifier(n_estimators=50, random_state=RANDOM_STATE)
ada_clf.fit(X_train, y_train)
y_pred_ada = ada_clf.predict(X_test)
acc_ada = accuracy_score(y_test, y_pred_ada)

gbm_clf = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=RANDOM_STATE)
gbm_clf.fit(X_train, y_train)
y_pred_gbm = gbm_clf.predict(X_test)
acc_gbm = accuracy_score(y_test, y_pred_gbm)

print(f"AdaBoost (n_estimators=50):")
print(f"  Accuracy: {acc_ada:.4f}")
print(f"\nGradient Boosting (n_estimators=100, max_depth=3):")
print(f"  Accuracy: {acc_gbm:.4f}")
print()

In [ ]:
# TODO 6: Entrenar XGBoost
print("TODO 6: Entrenar XGBoost")
print("=" * 60)

xgb_clf = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=RANDOM_STATE,
    use_label_encoder=False,
    eval_metric='logloss',
    verbosity=0
)
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)
acc_xgb = accuracy_score(y_test, y_pred_xgb)

print(f"XGBoost (n_estimators=100, max_depth=3):")
print(f"  Accuracy: {acc_xgb:.4f}")
print()

In [ ]:
# TODO 7: Tabla comparativa con cross-validation
print("TODO 7: Tabla comparativa (CV con 5 folds)")
print("=" * 60)

models = {
    'Decision Tree': dt_clf,
    'Random Forest': rf_clf,
    'AdaBoost': ada_clf,
    'Gradient Boosting': gbm_clf,
    'XGBoost': xgb_clf
}

cv_results = {}
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    cv_results[name] = {
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Test Accuracy': accuracy_score(y_test, model.predict(X_test))
    }

df_comparison = pd.DataFrame(cv_results).T
df_comparison.columns = ['cv_accuracy_mean', 'cv_accuracy_std', 'test_accuracy']
print(df_comparison.round(4))
print()

# Visualización
fig, ax = plt.subplots(figsize=(11, 6))
models_list = list(cv_results.keys())
means = [cv_results[m]['CV Mean'] for m in models_list]
stds = [cv_results[m]['CV Std'] for m in models_list]

ax.bar(models_list, means, yerr=stds, capsize=5, alpha=0.7, color=['red', 'green', 'blue', 'orange', 'purple'], edgecolor='black')
ax.set_ylabel('CV Accuracy (mean ± std)', fontsize=12, fontweight='bold')
ax.set_title('Comparación de Modelos (5-Fold Cross-Validation)', fontsize=13, fontweight='bold')
ax.set_ylim([0.8, 1.0])
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print()

In [ ]:
# TODO 8 [PhD]: Análisis de pesos y errores en AdaBoost
print("TODO 8 [PhD]: Análisis de pesos en AdaBoost")
print("=" * 60)

estimator_weights = ada_clf.estimator_weights_
estimator_errors = ada_clf.estimator_errors_

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Gráfico 1: Peso α_t
ax1.plot(range(len(estimator_weights)), estimator_weights, 'o-', linewidth=2, markersize=6, color='darkblue')
ax1.set_xlabel('Iteración (t)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Peso α_t', fontsize=11, fontweight='bold')
ax1.set_title('Evolución del peso α_t en AdaBoost', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Gráfico 2: Error ε_t
ax2.plot(range(len(estimator_errors)), estimator_errors, 's-', linewidth=2, markersize=6, color='darkred')
ax2.set_xlabel('Iteración (t)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Error ε_t', fontsize=11, fontweight='bold')
ax2.set_title('Evolución del error ε_t en AdaBoost', fontsize=12, fontweight='bold')
ax2.set_ylim([0, 0.5])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Primer α_t: {estimator_weights[0]:.4f}, Último α_t: {estimator_weights[-1]:.4f}")
print(f"Primer ε_t: {estimator_errors[0]:.4f}, Último ε_t: {estimator_errors[-1]:.4f}")
print()
print("Interpretación:")
print("  - Los α_t deberían decrecer si los modelos mejoran en generalización.")
print("  - Los ε_t deberían ser < 0.5 para que AdaBoost converja.")
print("  - Si ε_t ≈ 0.5, ese estimador es casi aleatorio y pesa poco.")
print()

In [ ]:
# Tests de sanidad PARTE 3 (NO MODIFICAR)
print("\n" + "="*60)
print("TESTS DE SANIDAD — PARTE 3")
print("="*60)

assert 0 <= acc_ada <= 1 and 0 <= acc_gbm <= 1 and 0 <= acc_xgb <= 1, "❌ Accuracies fuera de rango"
assert len(estimator_weights) == 50, f"❌ AdaBoost debería tener 50 pesos, tiene {len(estimator_weights)}"
assert all(0 < e < 0.5 for e in estimator_errors), "❌ Algunos errores fuera de (0, 0.5)"
assert len(df_comparison) == 5, "❌ Tabla de comparación debería tener 5 modelos"
print("✅ Todos los tests pasaron.")

# PARTE FINAL — Análisis e Interpretación

**Objetivo:** Reflexionar sobre el uso de ensembles en problemas reales.

**Tiempo:** 10 minutos | **Audiencia:** Diferenciado

---

In [ ]:
# REFLEXIONES Y ANÁLISIS POSTERIOR

print("="*60)
print("RESUMEN Y CONCLUSIONES")
print("="*60)
print()

# Tabla final comparativa
print("Tabla final de modelos entrenados:")
print(df_comparison.round(4))
print()

# Ganador
best_model = df_comparison['cv_accuracy_mean'].idxmax()
best_score = df_comparison['cv_accuracy_mean'].max()
print(f"Mejor modelo (CV): {best_model} con {best_score:.4f}")
print()

# BONUS — Voting y Stacking

**Opcional** — Ensembles avanzados con combinación explícita de modelos.

---

## Voting vs Stacking

**Voting:** Combina predicciones con votación (hard) o promedio ponderado (soft).

**Stacking:** Entrena un meta-learner que aprende a combinar las predicciones de los modelos base.

In [ ]:
# BONUS 1 [Pregrado]: VotingClassifier con soft voting
print("BONUS 1: VotingClassifier (soft voting)")
print("=" * 60)

voting_clf = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)),
        ('gbm', GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=RANDOM_STATE)),
        ('lr', LogisticRegression(max_iter=500, random_state=RANDOM_STATE))
    ],
    voting='soft'
)
voting_clf.fit(X_train, y_train)
y_pred_voting = voting_clf.predict(X_test)
acc_voting = accuracy_score(y_test, y_pred_voting)

print(f"Voting Classifier (RF + GBM + LR, soft voting):")
print(f"  Accuracy: {acc_voting:.4f}")
print()

In [ ]:
# BONUS 2 [Doctorado]: StackingClassifier
print("BONUS 2 [Doctorado]: StackingClassifier")
print("=" * 60)

stacking_clf = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)),
        ('gbm', GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=RANDOM_STATE))
    ],
    final_estimator=LogisticRegression(random_state=RANDOM_STATE),
    cv=5
)
stacking_clf.fit(X_train, y_train)
y_pred_stacking = stacking_clf.predict(X_test)
acc_stacking = accuracy_score(y_test, y_pred_stacking)

print(f"Stacking Classifier (base: RF, GBM; meta: LR, cv=5):")
print(f"  Accuracy: {acc_stacking:.4f}")
print()

# Comparación Voting vs Stacking
print("Comparación: Voting vs Stacking vs Mejor individual")
print("-" * 60)
best_individual_acc = df_comparison['test_accuracy'].max()
print(f"Mejor individual:  {best_individual_acc:.4f}")
print(f"Voting (soft):     {acc_voting:.4f} ({(acc_voting - best_individual_acc):+.4f})")
print(f"Stacking:          {acc_stacking:.4f} ({(acc_stacking - best_individual_acc):+.4f})")
print()

# 📋 Checklist de Entrega

## 🔵 Para Pregrado

- [ ] PARTE 1: Calculaste accuracy individual, hard voting, y explicaste por qué el ensemble (puede o no) supera
- [ ] PARTE 2: Entrenaste DT y RF, visualizaste top-10 features, graficaste learning curve
- [ ] PARTE 3: Entrenaste AdaBoost, GBM, XGBoost y creaste tabla comparativa con CV
- [ ] FINAL: Respondiste las 3 preguntas aplicadas (RF vs XGBoost, test contamination, explicación cliente)
- [ ] BONUS (opcional): Implementaste Voting Classifier y comparaste con modelos individuales

## 🟡 Para Doctorado

- [ ] Todas las tareas de Pregrado ✅
- [ ] PARTE 1: Calculaste matriz de errores, correlación de errores, teoría de Krogh & Vedelsby
- [ ] PARTE 2: Calculaste OOB score y comparaste con test accuracy
- [ ] PARTE 3: Analizaste evolución de α_t y ε_t en AdaBoost, interpretaste patrones
- [ ] FINAL: Respondiste las 3 preguntas analíticas (Ambiguity, bagging vs boosting, experimento de independencia)
- [ ] BONUS (opcional): Implementaste StackingClassifier y explicaste por qué puede superar a Voting

---

## ✅ Verificación final

Todos los tests de sanidad deberían pasar:
```
✅ Setup completo
✅ Tests PARTE 2
✅ Tests PARTE 3
```

---

## 🚀 Siguientes pasos

1. Experimenta con hiperparameters (learning_rate, n_estimators, max_depth)
2. Usa GridSearchCV para optimizar automáticamente
3. Aplica estos métodos a tus propios datasets
4. Lee papers de referencia: Breiman (2001) Random Forests, Schapire & Freund (2012) Boosting